# Đếm token: tiếng Việt tốn bao nhiêu so với tiếng Anh?

Giá API, giới hạn context và tốc độ sinh đều tính bằng **token**. Bài này đo trên **1 268 cặp câu Anh–Việt thật**
(tập test `tst2013` của IWSLT'15, phụ đề TED) xem cùng một nội dung thì bản tiếng Việt tốn bao nhiêu token với ba
tokenizer: `cl100k_base` (thế hệ GPT-4), `o200k_base` (thế hệ GPT-4o trở đi) và tokenizer của Qwen2.5 (model chạy trên
laptop của track). Rồi dùng con số đó để quản lý ngân sách context.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, tải tokenizer, in bảng; test bỏ qua chúng.
- Lần đầu chạy, cell `explore` tải tệp mã hoá của `tiktoken` (khoảng 4 MB mỗi bộ) và `tokenizer.json` của Qwen2.5
  (khoảng 7 MB) nên cần mạng; các lần sau dùng cache.
- Chấm: `uv run pytest tracks/ai-engineer/modules/00-introduction/exercises/intermediate/01-token-counting -q`

In [ ]:
# explore: tìm gốc repo, tải IWSLT'15 en-vi nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

IWSLT = ensure("iwslt15-en-vi")
print(IWSLT, sorted(p.name for p in IWSLT.iterdir()))

In [ ]:
import html
import re
from typing import Callable

Encode = Callable[[str], list]  # văn bản -> danh sách token id (chỉ cần len)

## 1. Bỏ tách từ

Tệp IWSLT đã được **tách từ** (tokenized) kiểu Moses: dấu câu đứng riêng, ký tự đặc biệt viết dạng HTML entity.
Dòng thật: `In school , we spent a lot of time ...` và `I can &apos;t ...`. Đếm token trên văn bản như vậy sẽ lệch so
với văn bản người dùng gõ, nên trước hết đưa về dạng tự nhiên.

`detokenize(line) -> str`, đúng thứ tự:

1. Giải HTML entity (`&apos;` thành `'`, `&quot;` thành `"`, `&amp;` thành `&`, `&#91;` thành `[`...).
2. Bỏ khoảng trắng **trước** các ký tự `, . ! ? ; : % ) ]`.
3. Bỏ khoảng trắng **sau** các ký tự `( [ $`.
4. Bỏ khoảng trắng trước dấu `'` khi ngay sau `'` là chữ cái Latin không dấu `A-Z`, `a-z` (`can 't` thành `can't`,
   `worker 's` thành `worker's`), giữ nguyên `' tự` (dấu nháy đứng trước khoảng trắng).
5. Gộp mọi chuỗi khoảng trắng thành một dấu cách, bỏ khoảng trắng đầu và cuối.

Dấu `"` giữ nguyên khoảng trắng hai bên (phân biệt mở và đóng nháy cần ngữ cảnh, không đáng công ở đây).

In [ ]:
def detokenize(line: str) -> str:
    # Bỏ tách từ kiểu Moses theo đúng 5 bước ở trên.
    s = html.unescape(line)
    s = re.sub(r"\s+([,.!?;:%)\]])", r"\1", s)
    s = re.sub(r"([(\[$])\s+", r"\1", s)
    s = re.sub(r"\s+'(?=[A-Za-z])", "'", s)
    return " ".join(s.split())

In [ ]:
# explore: đọc 1 268 cặp câu tst2013, bỏ tách từ, xem vài cặp
def read_lines(path: Path) -> list[str]:
    return path.read_text(encoding="utf-8").splitlines()


raw_en, raw_vi = read_lines(IWSLT / "tst2013.en"), read_lines(IWSLT / "tst2013.vi")
EN = [detokenize(s) for s in raw_en]
VI = [detokenize(s) for s in raw_vi]
assert len(EN) == len(VI) == 1268
for i in [0, 3, 22]:
    print("raw:", raw_en[i][:100])
    print("en: ", EN[i][:100])
    print("vi: ", VI[i][:100])
    print()

## 2. Thống kê token và "thuế token" của tiếng Việt

Một tokenizer ở đây chỉ là một hàm `encode(text) -> list` (chỉ dùng độ dài). Bạn sẽ so 6 cách "đếm": ký tự, byte
UTF-8, từ (tách theo khoảng trắng) làm tham chiếu, và 3 tokenizer thật.

- `token_stats(texts, encode) -> dict` với các khoá:
  `n_texts`, `n_chars` (tổng `len` ký tự), `n_words` (tổng số phần tử của `str.split()`), `n_tokens` (tổng
  `len(encode(t))`), `tokens_per_word = n_tokens / n_words`, `chars_per_token = n_chars / n_tokens`. Khi mẫu số bằng 0
  thì tỉ lệ là `0.0`.
- `token_tax(src_texts, tgt_texts, encode) -> float`: tổng token của `tgt_texts` chia tổng token của `src_texts`
  (ví dụ `src` tiếng Anh, `tgt` bản dịch tiếng Việt). Hai danh sách khác độ dài thì `ValueError`; tổng token của `src`
  bằng 0 thì `ValueError`.

In [ ]:
def token_stats(texts: list[str], encode: Encode) -> dict:
    # Tổng ký tự, từ, token và hai tỉ lệ tokens_per_word, chars_per_token.
    n_chars = sum(len(t) for t in texts)
    n_words = sum(len(t.split()) for t in texts)
    n_tokens = sum(len(encode(t)) for t in texts)
    return {
        "n_texts": len(texts),
        "n_chars": n_chars,
        "n_words": n_words,
        "n_tokens": n_tokens,
        "tokens_per_word": n_tokens / n_words if n_words else 0.0,
        "chars_per_token": n_chars / n_tokens if n_tokens else 0.0,
    }


def token_tax(src_texts: list[str], tgt_texts: list[str], encode: Encode) -> float:
    # Tổng token của bản dịch chia tổng token của bản gốc.
    if len(src_texts) != len(tgt_texts):
        raise ValueError("hai danh sách phải là các cặp câu, cùng độ dài")
    src = sum(len(encode(t)) for t in src_texts)
    if src == 0:
        raise ValueError("bản gốc không có token nào")
    return sum(len(encode(t)) for t in tgt_texts) / src

In [ ]:
# explore: 6 cách đếm, bảng so sánh (lần đầu tải tokenizer cần mạng)
import pandas as pd
import tiktoken
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

qwen = Tokenizer.from_file(hf_hub_download("Qwen/Qwen2.5-0.5B-Instruct", "tokenizer.json"))
ENCODERS: dict[str, Encode] = {
    "ký tự": list,
    "byte UTF-8": lambda s: s.encode("utf-8"),
    "từ (split)": str.split,
    "cl100k_base": tiktoken.get_encoding("cl100k_base").encode,
    "o200k_base": tiktoken.get_encoding("o200k_base").encode,
    "qwen2.5": lambda s: qwen.encode(s, add_special_tokens=False).ids,
}
rows = []
for name, enc in ENCODERS.items():
    en, vi = token_stats(EN, enc), token_stats(VI, enc)
    rows.append({"cách đếm": name, "token en": en["n_tokens"], "token vi": vi["n_tokens"],
                 "vi/en": round(token_tax(EN, VI, enc), 3),
                 "token/từ vi": round(vi["tokens_per_word"], 3), "ký tự/token vi": round(vi["chars_per_token"], 3)})
table = pd.DataFrame(rows).set_index("cách đếm")
display(table)

In [ ]:
# explore: nhìn tận mắt các mảnh token của một câu tiếng Việt, và phân phối tỉ lệ vi/en theo từng cặp câu
import matplotlib.pyplot as plt
import numpy as np

sample = VI[3]
for name in ["cl100k_base", "o200k_base", "qwen2.5"]:
    if name == "qwen2.5":
        pieces = [qwen.decode([i]) for i in qwen.encode(sample, add_special_tokens=False).ids]
    else:
        enc = tiktoken.get_encoding(name)
        pieces = [enc.decode([t]) for t in enc.encode(sample)]
    print(f"{name:12s} {len(pieces):3d} token:", pieces[:14])

fig, ax = plt.subplots(figsize=(7, 3.5))
for name in ["cl100k_base", "o200k_base", "qwen2.5"]:
    enc = ENCODERS[name]
    ratio = np.array([len(enc(v)) / max(1, len(enc(e))) for e, v in zip(EN, VI)])
    ax.hist(ratio, bins=np.linspace(0, 5, 51), alpha=0.5, label=f"{name} (trung vị {np.median(ratio):.2f})")
ax.set_xlabel("token tiếng Việt / token tiếng Anh, từng cặp câu")
ax.set_ylabel("số cặp")
ax.legend()
plt.show()

## 3. Quản lý ngân sách context

Context window chứa **cả** prompt lẫn phần model sinh ra. `qwen2.5:3b` qua Ollama mặc định dùng context 4 096 token.

- `fits_in_context(n_prompt, max_new_tokens, context_window) -> bool`: prompt cộng số token tối đa sẽ sinh không vượt
  context window.
- `truncate_to_budget(texts, encode, budget) -> list[str]`: lấy các đoạn **theo thứ tự** từ đầu, dừng ở đoạn **đầu
  tiên** làm tổng token vượt `budget` (không bỏ qua nó để nhét đoạn ngắn hơn phía sau: thứ tự thường mang nghĩa, ví dụ
  đoạn được xếp theo độ liên quan). Không cắt giữa một đoạn.

In [ ]:
def fits_in_context(n_prompt: int, max_new_tokens: int, context_window: int) -> bool:
    return n_prompt + max_new_tokens <= context_window


def truncate_to_budget(texts: list[str], encode: Encode, budget: int) -> list[str]:
    # Giữ tiền tố dài nhất của texts có tổng token <= budget.
    kept, used = [], 0
    for text in texts:
        n = len(encode(text))
        if used + n > budget:
            break
        kept.append(text)
        used += n
    return kept

In [ ]:
# explore: ngân sách 3 000 token cho tài liệu (4 096 context, chừa chỗ cho chỉ thị và câu trả lời)
BUDGET = 3000
for name in ["cl100k_base", "o200k_base", "qwen2.5"]:
    enc = ENCODERS[name]
    n_vi = len(truncate_to_budget(VI, enc, BUDGET))
    n_en = len(truncate_to_budget(EN, enc, BUDGET))
    print(f"{name:12s} chứa được {n_vi:4d} câu tiếng Việt, {n_en:4d} câu tiếng Anh")

prompt = "\n".join(truncate_to_budget(VI, ENCODERS["qwen2.5"], BUDGET))
n_prompt = len(ENCODERS["qwen2.5"](prompt))
print("prompt", n_prompt, "token; thêm 1 000 token trả lời có vừa 4 096?", fits_in_context(n_prompt, 1000, 4096))

## Đọc kết quả

- Theo ký tự, bản tiếng Việt dài hơn bản tiếng Anh bao nhiêu phần trăm? Theo token của từng tokenizer thì sao? Phần
  chênh nằm ở đâu: ở nội dung hay ở tokenizer?
- Vì sao "từ" tách theo khoảng trắng của tiếng Việt nhiều hơn tiếng Anh tới khoảng một phần ba, dù cùng nghĩa?
- Nếu một model tính 1 USD/MTok dùng `cl100k_base` và một model khác 1.5 USD/MTok dùng `o200k_base`, với khách hàng
  Việt Nam model nào rẻ hơn thật?
- Ghép câu bằng `"\n"` rồi đếm có ra đúng tổng token từng câu không? Vì sao chênh lệch nhỏ?

Sau đó làm phần trắc nghiệm của bài trong app.